In [3]:
pip install langchain==1.2.13 langchain-core==1.2.10 langchain-groq==1.1.2

  Using cached langchain-1.2.13-py3-none-any.whl.metadata (5.8 kB)
  Using cached langchain_core-1.2.10-py3-none-any.whl.metadata (4.4 kB)
  Using cached langchain_groq-1.1.2-py3-none-any.whl.metadata (2.4 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached langsmith-0.7.22-py3-none-any.whl.metadata (15 kB)
Using cached langchain-1.2.13-py3-none-any.whl (112 kB)
Using cached langchain_core-1.2.10-py3-none-any.whl (496 kB)
Using cached langchain_groq-1.1.2-py3-none-any.whl (19 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.1/168.1 kB 10.5 MB/s eta 0:00:00
Using cached langsmith-0.7.22-py3-none-any.whl (359 kB)
Using cached pydantic-2.12.5-py3-none-any.whl (463 kB)
  Attempting uninstall: pydantic
    Found existing installation: pydantic 1.10.13
    Uninstalling pydantic-1.10.13:
      Successfully uninstalled pydantic-1.10.13
  Attempting uninstall: langsmith
    Found existing installation: langsmith 0.1.81
    Uninstalling langsmith-0.1.81:
      

In [4]:
pip install langchain-huggingface

  Using cached langchain_huggingface-1.2.1-py3-none-any.whl.metadata (3.3 kB)
  Using cached langchain_core-1.2.22-py3-none-any.whl.metadata (4.4 kB)
Using cached langchain_huggingface-1.2.1-py3-none-any.whl (30 kB)
Using cached langchain_core-1.2.22-py3-none-any.whl (506 kB)
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.10
    Uninstalling langchain-core-1.2.10:
      Successfully uninstalled langchain-core-1.2.10
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-text-splitters 0.0.2 requires langchain-core<0.3,>=0.1.28, but you have langchain-core 1.2.22 which is incompatible.
langchain-community 0.0.38 requires langchain-core<0.2.0,>=0.1.52, but you have langchain-core 1.2.22 which is incompatible.
langchain-community 0.0.38 requires langsmith<0.2.0,>=0.1.0, but you have langsmith 0.7.22 which is incompati

In [5]:

from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests
from langchain_huggingface import HuggingFaceEmbeddings

# Tools Create


In [6]:
@tool
def multiply(a:int ,b : int) -> int:
  """Given 2 numbers a and b this tool returns their product"""
  return a * b

In [7]:
print (multiply.invoke({'a':3,"b":4}))


12


In [8]:
multiply.name

'multiply'

In [9]:
multiply.description

'Given 2 numbers a and b this tool returns their product'

In [10]:
multiply.args

{'a': {'title': 'A', 'type': 'integer'},
 'b': {'title': 'B', 'type': 'integer'}}

# Tool Binding

In [11]:
pip install --upgrade transformers

In [12]:
!pip install langchain-groq

In [13]:
pip install -U langchain langchain-core langchain-groq

In [14]:
from langchain_groq import ChatGroq

In [15]:
import os
os.environ["GROQ_API_KEY"] = "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"

In [16]:
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)

In [17]:
# tool create
from langchain_core.tools import InjectedToolArg
from typing import Annotated

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
  """
  This function fetches the currency conversion factor between a given base currency and a target currency
  """
  url = f'https://v6.exchangerate-api.com/v6/9d1880f3d452028bdda33e63/pair/{base_currency}/{target_currency}'

  response = requests.get(url)

  return response.json()

@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
  """
  given a currency conversion rate this function calculates the target currency value from a given base currency value
  """
  return base_currency_value * conversion_rate


In [18]:
convert.args

{'base_currency_value': {'title': 'Base Currency Value', 'type': 'integer'}}

In [19]:
get_conversion_factor.invoke({'base_currency':'USD','target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1774483201,
 'time_last_update_utc': 'Thu, 26 Mar 2026 00:00:01 +0000',
 'time_next_update_unix': 1774569601,
 'time_next_update_utc': 'Fri, 27 Mar 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 94.0283}

In [20]:
convert.invoke({'base_currency_value':10, 'conversion_rate':85.16})

851.5999999999999

In [21]:
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

In [22]:
messages = [HumanMessage('What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd')]

In [23]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd', additional_kwargs={}, response_metadata={})]

In [24]:
ai_message = llm_with_tools.invoke(messages)

In [25]:
messages.append(ai_message)

In [26]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'INR', 'target_currency': 'USD'},
  'id': 'frdcm911g',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': 10, 'conversion_factor': 0.014},
  'id': 'eqk8wwqsc',
  'type': 'tool_call'}]

In [27]:
import json

for tool_call in ai_message.tool_calls:
  # execute the 1st tool and get the value of conversion rate
  if tool_call['name'] == 'get_conversion_factor':
    tool_message1 = get_conversion_factor.invoke(tool_call)
    # fetch this conversion rate
    conversion_rate = json.loads(tool_message1.content)['conversion_rate']
    # append this tool message to messages list
    messages.append(tool_message1)
  # execute the 2nd tool using the conversion rate from tool 1
  if tool_call['name'] == 'convert':
    # fetch the current arg
    tool_call['args']['conversion_rate'] = conversion_rate
    tool_message2 = convert.invoke(tool_call)
    messages.append(tool_message2)


In [28]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'frdcm911g', 'function': {'arguments': '{"base_currency":"INR","target_currency":"USD"}', 'name': 'get_conversion_factor'}, 'type': 'function'}, {'id': 'eqk8wwqsc', 'function': {'arguments': '{"base_currency_value":10,"conversion_factor":0.014}', 'name': 'convert'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 47, 'prompt_tokens': 350, 'total_tokens': 397, 'completion_time': 0.087738676, 'completion_tokens_details': None, 'prompt_time': 0.031049626, 'prompt_tokens_details': None, 'queue_time': 0.160171712, 'total_time': 0.118788302}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_7ccc667439', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run-

In [29]:
llm_with_tools.invoke(messages).content

'The conversion factor between INR and USD is 0.01064. Therefore, 10 INR is equivalent to 0.1064 USD.'